In [ ]:
import logging

logging.basicConfig(level="DEBUG")
logging.getLogger("fsspec").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [ ]:
import torch
from darts.models import LinearRegressionModel

from darts.dataprocessing.transformers import Scaler
from sklearn.preprocessing import StandardScaler

from aare.feature_set import FeatureSet
from aare.features.registry import FEATURES

from aare.evaluation.evaluation import evaluate_model
from aare.params import read_params

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
params = read_params()

# Linear Regression

Even though we know that there are non-linear relationships at play, a simple linear regression model using the last water temperature, the relevant air temperature lag(s), as well as the time, might give a good comparison to other approaches.

In [ ]:
validation_params = params["validation"]
stride = validation_params["stride"]
min_lookback_hours = validation_params["min_lookback_hours"]
forecast_horizon = params["general"]["forecast_horizon"]

In [ ]:
ds = FeatureSet(
    targets=FEATURES["temp_bern"],
    future=[FEATURES["tt_bern"], FEATURES["tt_bern_log"], FEATURES["tt_bern_cube"], FEATURES["tt_bern_sqrt"]],
    split_params=params["split"],
)

In [ ]:
train = ds.get_train()
train

In [ ]:
val = ds.get_val()
val

In [ ]:
train_target_subs = train[0]
train_fc_subs = train[2]
val_target_subs = val[0]
val_fc_subs = val[2]

In [ ]:
[len(x) for x in train_target_subs]

In [ ]:
scaler_target = Scaler(StandardScaler(), global_fit=True)
scaler_fc = Scaler(StandardScaler(), global_fit=True)

In [ ]:
scaler_target.fit(train_target_subs)
scaler_fc.fit(train_fc_subs)

In [ ]:
from aare.evaluation.evaluation import DataTransformers

data_transformers: DataTransformers = {
    "series": scaler_target,
    "future_covariates": scaler_fc,
}

In [ ]:
model = LinearRegressionModel(
    # lags=[-1, -2],
    lags=1,  # use water temp at last hour
    # lags_future_covariates=[-2, -1, 0],
    lags_future_covariates=[-1],  # use air temp at last hour (highest corr)
    output_chunk_length=1,  # only predict 1 hour into the future
    # likelihood="quantile",
    quantiles=[0.25, 0.5, 0.75],
    random_state=42,
    multi_models=True,  # 1 model for each output step (doesn't matter if out_chunk_length is 1 anyway)
    use_static_covariates=False,  # currently no static covariates
    add_encoders={"cyclic": {"future": ["hour", "day_of_year"]}},
)

In [ ]:
model.fit(
    series=scaler_target.transform(train_target_subs),
    future_covariates=scaler_fc.transform(train_fc_subs),
)

In [ ]:
ex_val_target = val_target_subs[-1][28:30]
ex_val_fc = val_fc_subs[-1][: 32 + forecast_horizon]
ex_pred = model.predict(
    n=forecast_horizon,
    # num_samples=128,
    series=scaler_target.transform(ex_val_target),
    future_covariates=scaler_fc.transform(ex_val_fc),
)
ex_pred = scaler_target.inverse_transform(ex_pred)
ex_pred

In [ ]:
val_target_subs[-1][: 30 + forecast_horizon].plot(label="actual")
# ex_val_fc.plot(label="air temp")
ex_pred.plot(label="prediction")

In [ ]:
metrics, samples = evaluate_model(
    model,
    val_target_subs,
    forecast_horizon,
    stride,
    min_lookback_hours,
    future_cov=val_fc_subs,
    # num_samples=128,
    data_transformers=data_transformers,
)

In [ ]:
metrics

In [ ]:
_ = samples.plot("LR", with_covariates=["tt_bern"])

In [ ]:
_ = samples.plot("LR", with_covariates=False)